In [4]:
import os
import re
import json
import time
import math
import html
import argparse
import itertools
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup, NavigableString, Tag
from concurrent.futures import ThreadPoolExecutor, as_completed

# crawl thử ngày 1/1/2024

In [3]:
URL = "https://vietnamnet.vn/tin-tuc-24h?bydate=01%2F01%2F2024-01%2F01%2F2024&bydaterang=4&cate=000006"
headers = {"User-Agent": "Mozilla/5.0"}

# B1: lấy danh sách bài
html = requests.get(URL, headers=headers, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

items = []
for a in soup.select("h2 a, h3 a"):
    title = a.get_text(strip=True)
    href = urljoin(URL, a.get("href", ""))
    # lọc link bài viết .html
    if href.endswith(".html") and title:
        items.append({"title": title, "url": href})

# B2: vào trang chi tiết để lấy timestamp + nội dung (tối giản)
for it in items[:7]:  # demo vài bài đầu
    art = BeautifulSoup(requests.get(it["url"], headers=headers, timeout=30).text, "html.parser")
    # timestamp thường nằm gần tiêu đề; lấy text gần thẻ có định dạng ngày giờ
    time_tag = art.find(string=lambda x: x and ("GMT+07:00" in x or ":" in x))
    it["time_text"] = time_tag.strip() if time_tag else None

print(items)

[{'title': 'ĐH có an ninh nghiêm ngặt nhất Trung Quốc, giáo sư quên thẻ bị bảo vệ rượt đuổi', 'url': 'https://vietnamnet.vn/dh-an-ninh-nghiem-ngat-nhat-trung-quoc-giao-su-quen-the-bi-bao-ve-ruot-duoi-2232428.html', 'time_text': 'BEGIN COMPONENT:: COMPONENT002518'}, {'title': 'Đề kiểm tra cuối học kỳ 1 môn Toán lớp 10 Trường THPT Mạc Đĩnh Chi', 'url': 'https://vietnamnet.vn/de-kiem-tra-cuoi-hoc-ky-1-mon-toan-lop-10-truong-thpt-mac-dinh-chi-2232934.html', 'time_text': 'BEGIN COMPONENT:: COMPONENT002518'}, {'title': "Chính phủ 'chốt' điều chỉnh lộ trình tăng học phí đại học", 'url': 'https://vietnamnet.vn/chinh-thuc-tang-hoc-phi-dai-hoc-2233777.html', 'time_text': 'BEGIN COMPONENT:: COMPONENT002518'}, {'title': 'Thầy giáo về hưu khởi nghiệp, sở hữu khối tài sản 120.000 tỷ đồng ở tuổi 88', 'url': 'https://vietnamnet.vn/thay-giao-ve-huu-khoi-nghiep-tuoi-88-so-huu-khoi-tai-san-120-000-ty-dong-2233573.html', 'time_text': 'BEGIN COMPONENT:: COMPONENT002518'}, {'title': 'Đất nước nào đón năm mớ

# In ra nội dung ngày 1/1/2024 với tiêu mục education

In [10]:
URL = "https://vietnamnet.vn/tin-tuc-24h?bydate=02%2F01%2F2024-02%2F01%2F2024&bydaterang=4&cate=000006"
headers = {"User-Agent": "Mozilla/5.0"}

# B1: lấy danh sách bài
html = requests.get(URL, headers=headers, timeout=30).text
soup = BeautifulSoup(html, "html.parser")
soup

<!DOCTYPE html>

<html lang="vi" translate="no">
<head>
<!-- BEGIN COMPONENT:: COMPONENT002518 -->
<meta charset="utf-8"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type">
<meta content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=5, user-scalable=1" name="viewport"/>
<meta content="vietnamese" name="language">
<meta content="notranslate" name="google"/>
<!--Nhanh trang chu,  -->
<!--No ViewBag.NewsId -->
<!--No ViewBag.NewsId Dieuconmai-->
<title>Tin nhanh 24h online mới nhất</title>
<meta content="1800" http-equiv="Refresh"/>
<meta content="Tin nhanh 24h- Đọc báo online Việt Nam và thế giới 24h qua. Thông tin hình ảnh mới nhất trong ngày, tin thời sự, chính trị, xã hội 24h hôm nay" name="description"/>
<meta content="tin nhanh 24h, tin tức 24h, tin tuc 24h, tin nhanh 24h, tin tức 24h mới nhất, tin 24h, 24h" name="keywords"/>
<meta content="tin nhanh 24h, tin tức 24h, tin tuc 24h, tin nhanh 24h, tin tức 24h mới nhất, tin 24h, 24h" name="news_

# Thử in và lưu xuống file csv cả tiêu đề và nội dung

In [13]:
import csv
import re
import requests
from bs4 import BeautifulSoup, Tag

URL = "https://vietnamnet.vn/tin-tuc-24h?bydate=01%2F01%2F2024-01%2F01%2F2024&bydaterang=4&cate=000006"
headers = {"User-Agent": "Mozilla/5.0"}

def textnorm(x: str) -> str:
    return re.sub(r"\s+", " ", x).strip() if x else ""

def get_desc_for_title(title_tag: Tag) -> str:
    # Tìm trong cùng cụm cha gần nhất
    node = title_tag
    for _ in range(4):
        if not node or not isinstance(node, Tag):
            break
        cand = node.select_one(".horizontalPost__main-desc")
        if cand:
            return textnorm(cand.get_text(" "))
        node = node.parent
    # Fallback: lấy phần mô tả ngay sau đó (nếu có)
    cand = title_tag.find_next(class_="horizontalPost__main-desc")
    return textnorm(cand.get_text(" ")) if cand else ""

resp = requests.get(URL, headers=headers, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

# Lấy tất cả tiêu đề theo class đã nêu
title_tags = soup.select(".horizontalPost__main-title.vnn-title.title-bold")

rows = []
for idx, t in enumerate(title_tags, start=1):
    title_text = textnorm(t.get_text(" "))
    desc_text = get_desc_for_title(t)
    if not title_text:
        continue
    rows.append([idx, title_text, desc_text])

# Ghi CSV
out_path = "vietnamnet_tintuc24h.csv"
with open(out_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["thu_tu", "tieu_de", "noi_dung"])
    writer.writerows(rows)

print(f"Đã ghi {len(rows)} dòng vào {out_path}")


Đã ghi 7 dòng vào vietnamnet_tintuc24h.csv


# Crawl dữ liệu từ 12/4/2025 - 18/10/2025

In [3]:
# Crawl Vietnamnet theo yêu cầu:
# - Khoảng: 18/04/2025 -> 18/10/2025 (bao gồm)
# - Categories lặp theo ngày: mỗi ngày dùng 1 cate từ danh sách, khi tới cuối quay lại đầu
# - Lưu CSV **không có header**, **không có số thứ tự**, mỗi dòng chỉ chứa "nội dung" (title + " — " + desc)
# - Sử dụng tqdm để theo dõi tiến trình
#
# Yêu cầu: pip install requests beautifulsoup4 tqdm

import csv
import re
import time
from datetime import datetime, timedelta
from urllib.parse import quote

import requests
from bs4 import BeautifulSoup, Tag
from tqdm import tqdm

# --- Cấu hình ---
START = datetime(2025, 2, 15)
END   = datetime(2025, 4, 11)  # inclusive
BASE = "https://vietnamnet.vn/tin-tuc-24h"
HEADERS = {"User-Agent": "Mozilla/5.0"}
OUT_PATH = "gold-data.csv"
SLEEP_BETWEEN_REQUESTS = 0.6
RETRIES = 3

CATES = [
    "000006", "00000R", "00000E", "000001", "00MK8V", "000007", "00000U", "000003",
    "00000B", "000008", "00000W", "000005", "000009", "000002", "00M7F7", "00000T", "000004"
]
# -----------------

def textnorm(x: str) -> str:
    return re.sub(r"\s+", " ", x).strip() if x else ""

def build_url_for_day_and_cate(day: datetime, cate: str) -> str:
    d = day.strftime("%d/%m/%Y")
    # bydate expects dd/mm/yyyy-dd/mm/yyyy; quote sẽ encode '/' -> %2F
    bydate = quote(f"{d}-{d}", safe="-")
    return f"{BASE}?bydate={bydate}&bydaterang=4&cate={cate}"

def fetch_with_retries(session: requests.Session, url: str, retries: int = RETRIES, timeout: int = 30) -> str:
    last_exc = None
    for i in range(retries):
        try:
            r = session.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()
            return r.text
        except Exception as e:
            last_exc = e
            time.sleep(0.5 * (i + 1))
    raise last_exc

def get_desc_for_title(title_tag: Tag) -> str:
    # tìm .horizontalPost__main-desc trong cha gần nhất (độ sâu 4)
    node = title_tag
    for _ in range(4):
        if not node or not isinstance(node, Tag):
            break
        cand = node.select_one(".horizontalPost__main-desc")
        if cand:
            return textnorm(cand.get_text(" "))
        node = node.parent
    # fallback: tìm tiếp theo
    cand = title_tag.find_next(class_="horizontalPost__main-desc")
    return textnorm(cand.get_text(" ")) if cand else ""

def daterange(start: datetime, end: datetime):
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)

# --- Thực thi crawl ---
session = requests.Session()

with open(OUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    # Không ghi header theo yêu cầu

    days = list(daterange(START, END))
    pbar = tqdm(total=len(days), desc="Ngày", unit="day")

    for day_index, day in enumerate(days):
        # chọn cate luân phiên
        cate = CATES[day_index % len(CATES)]
        url = build_url_for_day_and_cate(day, cate)

        try:
            html = fetch_with_retries(session, url)
        except Exception as e:
            tqdm.write(f"[ERROR] {day:%d/%m/%Y} cate={cate} -> {e}")
            pbar.update(1)
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            continue

        soup = BeautifulSoup(html, "html.parser")
        title_tags = soup.select(".horizontalPost__main-title.vnn-title.title-bold")

        count_today = 0
        for t in title_tags:
            title_text = textnorm(t.get_text(" "))
            if not title_text:
                continue
            desc_text = get_desc_for_title(t)
            # Nội dung lưu: title + " — " + desc (nếu desc rỗng thì chỉ title)
            if desc_text:
                content = f"{title_text} — {desc_text}"
            else:
                content = title_text
            writer.writerow([content])
            count_today += 1

        pbar.set_postfix_str(f"{day:%d/%m/%Y} cate={cate} items={count_today}")
        pbar.update(1)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    pbar.close()

print(f"Done. Kết quả lưu tại: {OUT_PATH}")


Ngày:   0%|          | 0/56 [00:00<?, ?day/s]

Ngày: 100%|██████████| 56/56 [00:57<00:00,  1.03s/day, 11/04/2025 cate=00MK8V items=14]

Done. Kết quả lưu tại: gold-data.csv


# Chuyển đổi định dạng

In [2]:
# Convert the uploaded CSV into a .txt file in the format:
# 
# #1
# <review text>
#
# #2
# <review text>
#
# Heuristics to auto-detect the text column. The script prints out which column was chosen
# and saves the .txt for download.

import pandas as pd, os, re, json, numpy as np

src_csv = "./gold-data.csv"
out_txt = "./gold-data.txt"

# Try multiple read options for robustness
read_kwargs_options = [
    {"encoding": "utf-8"},
    {"encoding": "utf-8-sig"},
    {"encoding": "cp1258"},  # Vietnamese Windows-1258
    {"encoding": "cp1252", "errors": "ignore"},
]

df = None
last_err = None
for kw in read_kwargs_options:
    try:
        df = pd.read_csv(src_csv, **kw)
        break
    except Exception as e:
        last_err = e

if df is None:
    raise RuntimeError(f"Không thể đọc file CSV: {last_err}")

# Heuristic to choose the text column
candidates_priority = [
    "text","review","content","comment","sentence","review_text","body","message","desc","description",
    "noi_dung","binh_luan","nhan_xet","danh_gia"
]

cols_lower = {c.lower(): c for c in df.columns}

chosen_col = None
for key in candidates_priority:
    if key in cols_lower:
        chosen_col = cols_lower[key]
        break

if chosen_col is None:
    # Choose the first object-type column with the largest average string length
    obj_cols = [c for c in df.columns if df[c].dtype == object]
    if obj_cols:
        def avg_len(series):
            s = series.dropna().astype(str).str.strip()
            if len(s) == 0:
                return 0
            return s.str.len().mean()
        lengths = {c: avg_len(df[c]) for c in obj_cols}
        chosen_col = max(lengths, key=lengths.get) if lengths else obj_cols[0]
    else:
        # Fallback to the first column
        chosen_col = df.columns[0]

series = df[chosen_col].dropna().astype(str)

# Clean and keep non-empty
series = series.map(lambda s: re.sub(r'\s+', ' ', s.strip()))
series = series[series.str.len() > 0]

# Optional: drop duplicates while keeping order
series = series.drop_duplicates(keep="first")

# Compose the txt content
lines = []
for i, text in enumerate(series.tolist(), start=2012):
    lines.append(f"#{i}")
    lines.append(text)
    lines.append("")  # blank line between entries

content = "\n".join(lines)

with open(out_txt, "w", encoding="utf-8") as f:
    f.write(content)

# Print summary and a preview
print("Đã tạo file TXT đầu vào theo định dạng yêu cầu.")
print(f"Cột được sử dụng: {chosen_col}")
print(f"Số dòng đầu ra: {len(series)}")
print("\n--- Preview ---")
preview = content.splitlines()[:8]
print("\n".join(preview))

out_txt

Đã tạo file TXT đầu vào theo định dạng yêu cầu.
Cột được sử dụng: Bộ GD-ĐT: Năm 2025, các đại học không còn xét tuyển sớm — Năm 2025, các đại học sẽ không còn đợt xét tuyển sớm. Tất cả các phương thức sẽ được xét chung trong một đợt.
Số dòng đầu ra: 633

--- Preview ---
#2012
Đưa tiếng Anh trở thành ngôn ngữ thứ 2: Nỗ lực từ các trường đại học — Nâng cao kỹ năng tiếng Anh cho sinh viên tại các trường đại học có vai trò quan trọng trong mục tiêu “đưa tiếng Anh trở thành ngôn ngữ thứ hai trong trường học”, như nội dung trong Kết luận 91-KL/TW của Bộ Chính trị đã đặt ra.

#2013
Bài mẫu viết thư UPU lần thứ 54: Đại dương và nỗi ám ảnh khai thác khoáng sản — Chủ đề cuộc thi viết thư UPU lần thứ 54 được đánh giá khá thú vị khi người viết tưởng tượng mình là đại dương. VietNamNet xin giới thiệu bài mẫu, độc giả có thể tham khảo.

#2014
Ngôi trường tại Hải Phòng đạt chứng nhận quốc tế của CIS — Trường Tiểu học, THCS & THPT Vinschool Imperia ghi dấu ấn mới khi trở thành ngôi trường duy nhất tại

'./gold-data.txt'